# Análisis RFM

En las otras notebooks se define el proceso tomado hasta llegar acá. Mostramos la limpieza, la generación de las features RFM y sus trasnformaciones, elegimos varios modelos y seleccionamos sus hiperparametros a partir del resultado de sus métricas de validación interna.

Se corrieron muchas veces varios modelos. En la tabla inferior se muestran los mejores resultados para cada uno de ellos con 3 y 4 clusters, que son los valores que habíamos elegido anteriormente. (En el caso de DBSCAN ni siquiera lo incluimos debido a que los resultados eran sumamente inferiores al resto.)

| Método       | Clusters |   AD   |  ADM   |   APN   | Calinski-Harabasz | Davies-Bouldin |  Dunn  |   FOM   | Silhouette |
|--------------|----------|--------|--------|---------|--------------------|----------------|--------|---------|------------|
| Agglomerative|    3     | 0.2528 | 0.4337 | 0.2909  |     4364.84        |     0.7476     | 0.0108 | 0.2833  | 0.4232     |
|              |    4     | 0.2232 | 0.4256 | 0.7425  |     5505.36        |     0.8075     | 0.0108 | 0.2557  | 0.4278     |
| Gaussian     |    3     | 0.2991 | 0.0878 | 0.7928  |     3204.45        |     0.8768     | 0.0030 | 0.2955  | 0.3948     |
|              |    4     | 0.2641 | 0.3320 | 0.6909  |     3383.61        |     0.9575     | 0.0013 | 0.2552  | 0.3558     |
| KMeans       |    3     | 0.2268 | 0.3462 | 0.7930  |     6875.30        |     0.6636     | 0.0024 | 0.2753  | 0.5158     |
|              |    4     | 0.2132 | 0.3056 | 0.7487  |     6768.43        |     0.7988     | 0.0026 | 0.2487  | 0.4800     |

El método con los mejores resultados parece ser KMeans. Aunque hay algunas métricas en donde otros lo superan, como por ejemplo el APN de Agglomerative con 3 clusters. Sin embargo, en la mayoría de las métricas KMeans es el mejor por diferencias en algunos casos amplias, y si queda por debajo generalmente es por poco.

Por otro lado, comparando 3 y 4 clusters de KMeans tenemos una decisión díficil. Ambos tienen algunas buenas métricas que superan al otro. Para las métricas más referidas al agrupamiento, tres clusters parece ser sutilmente mejores. Para las métricas de estabilidad, cuatro clusters hacen lo propio.

En este caso vamos a tomar 4 clusters, debido a que consideramos que nos da clusters un poco más representativos con una simple mirada a los boxplots que generabamos para las variables en el pipeline. Permitiendo que no todos los grandes monetaries se junten en un sólo cluster.

In [ ]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

#Cargamos los datos para el análisis
df = rfm_clusters.copy()
df.set_index('CustomerID', inplace=True)
df.head()

A los valores de RFM originales vamos a aplicarles Winsorization sin RobustScaler para facilitar las visualizaciones y evitar ver esos clientes extremadamente únicos. Además, agreguegamos algunas variables que nos pueden servir para hacer un análisis posterior.

In [ ]:

def winsorize_by_percentile(data, lower_percentile=5, upper_percentile=95):
    """
    Aplica winsorización a un DataFrame o a una Serie de Pandas.

    Los valores por debajo del percentil inferior se reemplazarán por el valor
    del percentil inferior. Los valores por encima del percentil superior se
    reemplazarán por el valor del percentil superior.

    Parámetros:
    -----------
    data : pd.DataFrame o pd.Series
        El conjunto de datos al que se aplicará la winsorización.
        Si es un DataFrame, la winsorización se aplica columna por columna.
        Si es una Serie, se aplica a la Serie.
    lower_percentile : int o float, opcional (default=5)
        El percentil inferior (e.g., 5 para el 5to percentil).
        Debe estar entre 0 y 100.
    upper_percentile : int o float, opcional (default=95)
        El percentil superior (e.g., 95 para el 95to percentil).
        Debe estar entre 0 y 100.

    Retorna:
    --------
    pd.DataFrame o pd.Series
        Los datos con la winsorización aplicada.
    """

    if not (0 <= lower_percentile < upper_percentile <= 100):
        raise ValueError("Los percentiles deben estar entre 0 y 100, "
                         "y lower_percentile debe ser menor que upper_percentile.")

    # Convertir a DataFrame si la entrada es una Serie para manejar ambos casos uniformemente
    if isinstance(data, pd.Series):
        is_series = True
        df = data.to_frame()
    elif isinstance(data, pd.DataFrame):
        is_series = False
        df = data.copy() # Trabajar con una copia para no modificar el DataFrame original
    else:
        raise TypeError("La entrada debe ser un pd.DataFrame o pd.Series.")

    winsorized_df = pd.DataFrame(index=df.index, columns=df.columns)

    for column in df.columns:
        # Asegurarse de que la columna sea numérica
        if pd.api.types.is_numeric_dtype(df[column]):
            lower_bound = np.percentile(df[column].dropna(), lower_percentile)
            upper_bound = np.percentile(df[column].dropna(), upper_percentile)

            winsorized_col = df[column].clip(lower=lower_bound, upper=upper_bound)
            winsorized_df[column] = winsorized_col
        else:
            # Si no es numérica, simplemente copiar la columna
            winsorized_df[column] = df[column]

    return winsorized_df.iloc[:, 0] if is_series else winsorized_df

original_rfm = rfm_data.copy()
original_rfm = winsorize_by_percentile(original_rfm, lower_percentile=10, upper_percentile=90)
original_rfm['Cluster'] = df['Cluster']
original_rfm.head()

PCA nos va ayudar a ver como se han formado los grupos.

In [ ]:
def plot_pca_clusters_figure(pca_df: pd.DataFrame, explained_variance: list, x_col="PC1", y_col="PC2", cluster_col="Cluster") -> plt.Figure:
    """
    Genera una figura de Matplotlib con los clusters visualizados en el espacio PCA.

    Parámetros:
    - pca_df: DataFrame con columnas para PC1, PC2 y la columna de clusters.
    - x_col, y_col: nombres de las columnas para los ejes x e y.
    - cluster_col: nombre de la columna que contiene la asignación de clusters.

    Retorna:
    - fig: objeto matplotlib.figure.Figure
    """
    # Crear figura y ejes
    fig, ax = plt.subplots(figsize=(8, 6))

    # Graficar cada cluster
    for cluster in sorted(pca_df[cluster_col].unique()):
        subset = pca_df[pca_df[cluster_col] == cluster]
        ax.scatter(subset[x_col], subset[y_col], label=f"Cluster {cluster}", alpha=0.6)

    # Configurar ejes y leyenda
    ax.set_title("Clusters visualizados en espacio PCA")
    ax.set_xlabel(f'{x_col} ({explained_variance[0]})')
    ax.set_ylabel(f'{y_col} ({explained_variance[1]})')
    ax.legend()
    ax.grid(True)

    return fig

def get_pca(data: pd.DataFrame, n_components: int):
    """
    Toma los datos y los reduce a n_components componentes principales.

    Parámetros:
    - df: DataFrame con columnas para PC1, PC2 y la columna de clusters.
    - n_components: número de componentes principales.

    Retorna:
    - fig: objeto matplotlib.figure.Figure
    - sum_explained_variance: suma de los explained_variance_ratio_
    """
    from sklearn.decomposition import PCA
    df = data.copy()
    df.drop(columns=['Cluster'], inplace=True)
    pca = PCA(n_components=n_components)
    pca_data = pca.fit_transform(df)
    pca_results = pd.DataFrame(pca_data, columns=[f'PC{i+1}' for i in range(n_components)], index=data.index)
    pca_results['Cluster'] = data['Cluster']
    print(pca_results.head(5))
    fig = plot_pca_clusters_figure(pca_results, pca.explained_variance_ratio_)
    sum_explained_variance = sum(pca.explained_variance_ratio_)
    return fig, sum_explained_variance

get_pca(df, 2)

Como podemos ver en el PCA, KMeans formó cuatro clusters bastante bien definidos, donde no se ve demasiada superposición salvo por algunos puntos. Además los dos componentes principales están representando bastante bien la varianza de nuestras features RFM.

Vamos a ver los violinplots de las features para entender esta separación en términos de RFM. 

In [ ]:
def plot_violinplots(df, cluster_col: str, max_cols=3):
    """
    Crea un grid de violin plots para cada variable numérica en el DataFrame.

    Parámetros:
    -----------
    df : pandas.DataFrame
        El DataFrame con las variables a graficar.
    figsize : tuple
        Tamaño total de la figura.
    max_cols : int
        Número máximo de columnas en el grid de subplots.
    """
    numeric_cols = df.select_dtypes(include=np.number).columns
    numeric_cols = np.delete(numeric_cols, numeric_cols.get_loc(cluster_col))
    n_vars = len(numeric_cols)
    
    if n_vars == 0:
        print("No hay variables numéricas para graficar.")
        return

    n_cols = min(n_vars, max_cols)
    n_rows = math.ceil(n_vars / n_cols)
    figsize = (8 * n_cols, 8 * n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)  # Asegura que axes sea plano

    for i, col in enumerate(numeric_cols):
        sns.violinplot(data=df, y=col, x=cluster_col, ax=axes[i])
        axes[i].set_title(f'Boxplot: {col}')
        axes[i].set_xlabel('Cluster')
    
    # Oculta subplots vacíos
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

plot_violinplots(original_rfm, 'Cluster')

Vamos a analizar que se ve en los gráficos.

#### Recency

Recency representa la cantidad de días que pasaron desde que el cliente realizó su última compra (desde el final del dataset). Por lo tanto valores más bajos son convenientes para el negocio. Un cliente con Recency alto indica que no compra hace mucho tiempo y posiblemente lo estamos perdiendo. Un cliente con recency bajo es un cliente nuevo o un cliente muy activo.

En cuanto a los clusters definidos con KMeans podemos ver lo siguiente:

- Cluster 0: La Recency es baja, con una mediana un poco inferior a 50 días. El 75% de los clientes de este clúster compraron hace menos de 75 días. Son clientes recientes a moderadamente recientes.
- Cluster 1: La Recency es también baja, con una mediana incluso menor al anterior, cerca de los 25 días. El 75% de los clientes compraron hace menos de 60 días. Sin embargo, a diferencia del cluster anterior hay algunos outliers que llegan casi a los 250 días. Esta varianza nos puede indicar que hay otra variable más fuerte en este cluster.
- Cluster 2: La Recency es muy alta, con una mediana superior a 200 días y cerca de los 250. Este clúster tiene los clientes menos recientes, y por lo tanto los que están en rojo en términos de riesgo.
- Cluster 3: La Recency es muy baja, la menor de las cuatro, con una mediana por debajo de 25 días. Son los clientes más recientes. Sin embargo, parecido al cluster uno existen bastantes outliers que nos pueden estar indicando que el "fuerte" de este cluster está en otro lado.

#### Frequency

Frequency nos indica la cantidad de veces que un cliente compró durante el periodo analizado. Es siempre un número entero y muchas veces se repite en los valores más bajo, razón por la que los outliers de los clusters se amontonan en ciertos valores.

En cuanto a los clusters definidos podemos decir lo siguiente:

- Cluster 0: La Frequency es muy baja, con una mediana de tan sólo una compra y el 75% de los clientes compraron únicamente dos veces o menos. Son clientes de baja frecuencia.
- Cluster 1: En este caso vemos algo muy especial, la mediana cae exactamente en el medio del rango de valores y los bigotes se extienden del comienzo al final. La distribución se ve bastante normal pero aplanada.
- Cluster 2: Ídem cluster cero con algunos outliers que llegan a 6.
- Cluster 3: Son los clientes de más alta frecuencia, la mediana está en el valor más alto que había sido fijado como límite. Estamos ante nuestros clientes más fieles. Se destaca algunos outliers que llegan hasta una frecuencia de uno. Es decir, el cluster no está completamente definido por la frecuencia, pero parece ser importante.

#### Monetary

Monetary se refiere al monto total que gastaron los clientes en nuestro negocio, en este caso en libras. Queremos que sea lo más alto posible.

Definiendo los clusters:

- Cluster 0: Monetary relativamente bajo, con una mediana cercana a 500 pero por debajo de este valor. El 75% de los clientes de este cluster no llegaron a gastar más de 750 libras.
- Cluster 1: Vemos algo parecido a frequency, un boxplot extenso que ocupa casi todo el rango de valores. La mediana está en 1500 y hay outliers por arriba de 3000.
- Cluster 2: El Monetary es muy bajo, con una mediana por debajo de 500. Son los clientes de menor gasto.
- Cluster 3: El Monetary es extremadamente alto, con una mediana en nuestro mayor valor. Incluso el primer quartil empieza donde termina el rango interquantil superior del cluster uno. Son con diferencia los clientes que más gastaron en nuestro negocio.

## Análisis de clusters

Vamos a ver a cada cluster de manera mas detallada. Intentaremos encontrar cosas que los describan aparte del RFM para lograr mejores Insights.

Para ello vamos a agregar al dataset algunas variables extra.

In [ ]:
original_rfm['avg_prod_per_invoice'] = (
    preprocessed_data.groupby(['CustomerID', 'InvoiceNo'])
    .size()
    .reset_index(name='LineasDeVenta')
).groupby('CustomerID')['LineasDeVenta'].mean()
original_rfm['avg_invoice_total'] = original_rfm['Monetary'] / original_rfm['Frequency']
original_rfm['avg_quantity'] = preprocessed_data.groupby('CustomerID')['Quantity'].mean()
original_rfm['categories_percentage'] = preprocessed_data.groupby('CustomerID')['Category'].nunique() / preprocessed_data['Category'].nunique()
original_rfm['Country'] = preprocessed_data.groupby('CustomerID')['Country'].first()
original_rfm.head()

#### Cluster 0 - Base de minoristas activos y nuevos clientes.

In [ ]:
cluster_zero = original_rfm[original_rfm['Cluster'] == 0]
cluster_zero.describe()

In [ ]:
zero_sales = preprocessed_data[preprocessed_data['CustomerID'].isin(cluster_zero.index)]
zero_sales.shape

In [ ]:
zero_sales['Category'].value_counts().plot(kind='bar')

In [ ]:
zero_sales['Month'].value_counts().sort_index().plot(kind='bar')

In [ ]:
zero_sales['InvoiceDate'].dt.dayofweek.value_counts().sort_index().plot(kind='bar')

In [ ]:
zero_sales['InvoiceDate'].dt.hour.value_counts().sort_index().plot(kind='bar')

In [ ]:
cluster_zero['Country'].value_counts().plot(kind='bar')

Ninguna de las nuevas variables parecen mostrar algo sumamente significativo. A partir de avg_quantity y avg_product_per_invoice podemos deducir que estamos hablando de un cluster con clientes minoristas y unos cuantos mayoristas recién llegados.

Lo que más caracteriza a este cluster es la recencia y frecuencia bajas. Que indican clientes nuevos, o minoristas casuales que compraron más de una vez y lo hicieron recientemente.

Son clientes que son de bajo valor, no nos brindan demasiadas ganancias por separado, pero que son mayoría. Representan 1751 de nuestros 4333 clientes. Por lo que, a pesar de no ser los mejores, la suma de todos ellos es necesaria para poder mantener el negocio.

Exploran poco, tienen una baja tasa de categorías diferentes compradas. Además, la mayoría de compras se producen en los últimos meses del año. Esto es lo normal, en general sucede así, pero en este caso la diferencia varía demasiado. También tiene que ver con la cuestión de que son los clientes recientes y el final del dataset es justo a fin de año. Pero cabe recalcar que muchas veces estamos hablando de eventos importantes como navidad, halloween, black friday o el comienzo de las clases en Europa que se da en septiembre.

#### Cluster 1 - Pequeños mayoristas y minoristas VIP

In [ ]:
cluster_one = original_rfm[original_rfm['Cluster'] == 1]
cluster_one.describe()

In [ ]:
one_sales = preprocessed_data[preprocessed_data['CustomerID'].isin(cluster_one.index)]
one_sales.shape

In [ ]:
one_sales['Category'].value_counts().plot(kind='bar')

In [ ]:
one_sales['Month'].value_counts().sort_index().plot(kind='bar')

In [ ]:
one_sales['InvoiceDate'].dt.dayofweek.value_counts().sort_index().plot(kind='bar')

In [ ]:
one_sales['InvoiceDate'].dt.hour.value_counts().sort_index().plot(kind='bar')

In [ ]:
cluster_one['Country'].value_counts().plot(kind='bar')

Este cluster en términos de RFM es muy especial. No es un cluster muy bien definido en términos de frecuencia o de monetary. Es cierto que al ver las distribuciones la mayoría se ubican con una frecuencia y monerary más altas que los clusters cero y dos, pero más bajas que las del cluster tres. Esto nos indica que el cluster se trata de clientes de mayor valor que los mencionados, pero que no llegan a ser los mejores.

Analizando la cantidad de productos por factura y la cantidad de productos promedio ligeramente superiores a los cluster cero y dos podemos llegar a entender que se trata de un cluster con mayoristas pequeños o que están en proceso de convertirse en clientes del cluster 3. 

Cuenta con algo de ruido, clientes que podrían haber quedado en otros cluster tranquilamente (razón por la que vemos esas colas largas en las distribuciones). Esto tiene algo de sentido, si miramos el PCA nos damos cuenta que el Cluster 1 es el que se formó exactamente en el medio, y al no estar muy bien separados hubo algunos puntos que se asociaron al centroide incorrecto. El gráfico de Silhouette mostrado abajo es una muestra de ello. Por esa razón, podemos encontra algunos minoristas VIP o principios de mayoristas VIP en este cluster.

A pesar de ese ruido consideramos a este cluster como mayoristas pequeños porque parecen representar el grueso de los datos. Estos tienen una menor tasa de exploración de categorías que los mayoristas del cluster tres. Esto puede darse por dos razones: son clientes especializados en algunos rubros o todavía no compraron las veces suficientes para mejorar ese score.

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score

def plot_silhouette(X, labels):
    silhouette_vals = silhouette_samples(X, labels)
    silhouette_avg = silhouette_score(X, labels)
    n_clusters = len(set(labels))
    
    fig, ax = plt.subplots(figsize=(8, 5))
    y_lower = 10
    for i in range(n_clusters):
        ith_cluster_silhouette_values = silhouette_vals[labels == i]
        ith_cluster_silhouette_values.sort()
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = sns.color_palette("hsv", n_clusters)[i]
        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7
        )
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10

    ax.set_title("Silhouette Plot")
    ax.set_xlabel("Silhouette Coefficient Values")
    ax.set_ylabel("Cluster")
    ax.axvline(x=silhouette_avg, color="red", linestyle="--")
    ax.set_yticks([])
    
    fig.show()

plot_silhouette(df.drop(columns=['Cluster']), df['Cluster'])

#### Cluster 2 - Clientes perdidos o en riesgo

In [ ]:
cluster_two = original_rfm[original_rfm['Cluster'] == 2]
cluster_two.describe()

In [ ]:
two_sales = preprocessed_data[preprocessed_data['CustomerID'].isin(cluster_two.index)]
two_sales.shape

In [ ]:
two_sales['Category'].value_counts().plot(kind='bar')

In [ ]:
two_sales['Month'].value_counts().sort_index().plot(kind='bar')

In [ ]:
cluster_two['Country'].value_counts().plot(kind='bar')

En este caso hablamos de clientes que tienen en común algo muy importante, un muy bajo recency. Esto nos indica que los clientes de este cluster están perdidos o en riesgo. Deberíamos buscar métodos que nos ayuden a retenerlos.

La mayor concentración en monetary y frecuencias bajas nos indican que el grueso de este tipo de clientes son minoristas casuales. Clientes que compraron alguna vez por un motivo en específico, nunca volvieron y posiblemente ni siquiera se acuerdan de nuestra existencia en este punto. 

Sin embargo también existen algunos clientes de alto monetary o frecuencia que están en este cluster. Ellos son los que más nos importan, eran clientes fieles (algunos mayoristas que habían gastado bastante) y que por algún motivo dejaron de confiar en nosotros.

Obviamente las compras de ellos se ubican en los primeros meses del año, que son las fechas más alejadas del final del dataset.

#### Cluster 3

In [ ]:
cluster_three = original_rfm[original_rfm['Cluster'] == 3]
cluster_three.describe()

In [ ]:
three_sales = preprocessed_data[preprocessed_data['CustomerID'].isin(cluster_three.index)]
three_sales.shape

In [ ]:
three_sales['Category'].value_counts().plot(kind='bar')

In [ ]:
three_sales['Month'].value_counts().sort_index().plot(kind='bar')

In [ ]:
three_sales['InvoiceDate'].dt.dayofweek.value_counts().sort_index().plot(kind='bar')

In [ ]:
three_sales['InvoiceDate'].dt.hour.value_counts().sort_index().plot(kind='bar')

In [ ]:
cluster_three['Country'].value_counts().plot(kind='bar')

En este cluster encontramos los clientes de mayor gasto y en su gran mayoría los más frecuentes. Estos clientes tienen que ser considerados prácticamente como aliados. Es el cluster con menor cantidad de clientes pero cuenta con más del doble de líneas de ventas que los otros clusters. Son los que más gastaron, los que más frecuente compran y en su mayoría siguen activos, habiendo comprado hace poco.

Estos clientes son con los que debemos tener más cuidado. El promedio de productos por factura, la cantidad de productos individuales y los totales de las facturas nos dan el indicio de que probablemente estemos hablando de clientes mayoristas que se nutren de nuestra web para la reventa.

Compran bastantes productos de varias categorías, lo que se podría llegar a explicar por su gran cantidad de compras. Son tiendas posiblemente con local físico estilo bazar, donde se venda todo este tipo de cosas variadas. Pero no son empresas exacatemnte chicas, deben ser locales más o menos grandes con muchas ventas.

También tienen una tendencia a comprar más en los últimos meses del año, que es donde deben suministrarse bien para las fiestas.

## Estrategias

#### Cluster 0 - Base de minoristas activos y nuevos clientes.

En este cluster queremos aumentar la frecuencia de compra, incrementar el gasto promedio por transacción y fomentar la exploración de más categorías de productos. Queremos que los clientes más nuevos tengan además un incentivo para comprar otra vez, que nos sigan eligiendo.

Algunas acciones de marketing que podríamos tomar son:
- Otorgar cupones de descuento luego de la primera compra. De esta forma incentivamos que aquellos clientes que nos compran por primera vez tienen una razón para volver a hacerlo. Al dar un descuento, si la persona había entrado por algo en específico, también estamos haciendo que el cliente miré más de nuestros productos. "No necesito nada en específico pero capaz hay algo con lo que pueda aprovechar el descuento" hace que el cliente miré más cosas que le interesan, gasté más o vuelva en otra ocasión.
- Sugerir productos complementarios a sus compras anteriores. Esto es clave para los minoristas, si compran cucharitas de té hay que mostrarles también las tacitas para que termine de armar el juego. Esto hará que el cliente siga viendo productos que le interesan y posiblemente gaste más o vuelva a comprar.
- Enviar mails recurrentes haciendo saber las ofertas por fiestas o eventos como el comienzo de clases. A todos nos gustan los decuentos, pero el minorista muchas veces solo compra si los hay. A diferencia de los mayoristas, los minoristas no tienen un stock que cubrir, y si no tienen urgencia pueden esperar. El problema es que muchas veces el cliente se olvida de nosostros entre oferta y oferta, o nunca se entera de que las hubo. Por ese motivo debemos tener especial insistencia con estos clientes cuando se habla de ofertas.
- Crear packs de productos para aumentar el valor de compra. Esto no solo nos permite aumentar el monetary de los minoristas, que siempre se verán atraídos a comprar el pack con un pequeño descuento al producto por separado. El valor de los clientes aumentará y también la rotación de productos. Podríamos usar items frecuentes para generar estos packs.

#### Cluster 1 - Pequeños mayoristas y minoristas VIP

Nuestra mayor preocupación para estos pequeños mayoristas es la fidelización. Queremos aumentar su frecuencia de compra hasta que se conviertan en clientes del cluster tres. Quizás algunos no llegarán, pero debemos estar atentos a aquellos que si pueden y darles los incentivos necesarios para lograrlo.

Las acciones de marketing podrían resumirse en:

- Fijar metas. Queremos estar atentos a aquellos clientes de este cluster que pueden llegar a ser un cliente VIP, por esa razón podemos fijar metas que nos indiquen si está progresando. Estas metas podrían ser internas, únicamente como métrica, o podemos hacerle saber al cliente sobre ella y otorgarle descuentos al conseguirlas. Quizá simplemente podemos mencionarle las condiciones necesarias para empezar a tener un trato "diferenciado" como el de los clientes del cluster tres. Es decir, hacer un sistema de lealtad por niveles.
- Muestras de productos. Si queremos convertir a los clientes del cluster 1 en clientes del cluster 3 debemos aumentar la variedad de categorias que compran. Regalar muestras de algunos productos de otras categorías que no compraron podría mostrarles el potencial de la diversificación.
- Recomendar productos con mayor rotación. Podemos recomendar a este tipo de clientes comprar productos que sepamos que tienen una alta rotación. Si todas las recomendaciones van orientadas hacia eso nos aseguramos de que usen sus presupuestos más limitados en esos productos que se venden rápido. Esto puede generar que compren más frecuentemente para reponer stock y generen mayor fidelidad a nuestro negocio. Una pequeña orientación para que crezcan más rápido junto a nosotros. 

#### Cluster 2 - Clientes perdidos o en riesgo

El principal objetivo de este cluster debería ser intentar re-captar a esos clientes perdidos o, aunque sea, intentar entender porque nos abandonaron para mejorar nuestras estrategias y trato al cliente.

Algunas acciones de marketing posibles son:

- Descuentos para clientes inactivos. Podemos intentar contactarlos con una oferta de descuento lo suficientemente atractiva para que se interesen en comprar nuevamente. Además con este descuento promovemos que los minoristas, en mayor medida, entren a "chusmear" que hay de nuevo logrando que quizás se enganchen con algún producto.
- Encuestas de abandono. Si no nuestro intento de atraerlos con descuentos falla por lo menos queremos saber porque nos dejaron de comprar. Podemos hacer una pequeña encuesta preguntando los motivos. Obviamente la misma debe ser sumamente corta y al grano para obtener más respuestas. Probablemente no obtengamos muchas respuestas, pero aquellos clientes muy enojados por algo seguramente si la hagan. Esto nos permitirá encontrar algunos de nuestros mayores defectos y las razones por la que alguien puede abandonarnos.
- Contacto directo con clientes de alto valor perdidos. Esto es muy importante, con los minoristas casuales podemos pensar que son ciclicos, pero los clientes de alto valor que nos dejaron son un problema. Con ellos estrategias de descuentos y encuestas quedan cortas, deberíamos intentar contactarlos directamente y tener una conversación acerca de que pasó. De esta conversación debemos obtener claramente las razones y generar planes de venta específicos para ellos, que los motiven a volver a comprarnos.
- Pedir al menos dos medios de contacto. No sabemos si esto entra en marketing exactamente, pero quizás los clientes que no vuelven es porque nuestros contactos con ellos fallan. Quizás cambiaron la cuenta de mail o la escribieron mal. Contar con dos medios de contacto puede reducir estos errores y hacer la comunicación de ofertas más efectiva.

#### Cluster 3 - Clientes mayoristas VIP

Estos clientes son los que no podemos perder, ya dijimos, deberían ser prácticamente considerados aliados. Son los que mayores ganancias nos generan y los que más nos compran. Las estrategias deben estar orientadas a hacerlos sentir lo más cómodos posibles para que no exista la idea de pasarse a la competencia.

Algunas de estas acciones pueden ser:

- Canal de comunicación exclusivo. Deberíamos contar con vendedores específicos a los cuales asignarles estos clientes. Los mismos deberían tener contacto directo constantemente para poder asegurarnos de que el cliente se mantiene contento. La simple página web no es suficiente para este tipo de clientes, necesitamos estar mejor coordinados con ellos. Algunos de ellos quizás necesiten algún trato especial o diferente atención que el vendedor pueda darles. Mejora además el servicio post venta que podamos darle.
- Ofertas por volumen. Claramente podemos otorgar descuentos más significativos a estos clientes que compran en grandes cantidades, para darles precios más atractivos que la competencia.
- Contacto antes de eventos importantes. Al igual que nosotros estas tiendas deben vender más en temporadas de fiestas o eventos. Podemos contactarlos antes de cada una para coordinar mejor los envíos y contarles de los nuevos productos disponibles (en caso de las fiestas por ejemplo).
- Brindar información y asesoría. Podemos generar informes para nuestros clientes mayoristas según nuestros propios datos. Podemos sugerirles que comprar según lo que está de moda y lo que nosotros más estamos vendiendo. Compartir este tipo de información le da un valor agregado a nuestro negocio, ya que no solo le damos productos o buena atención, también le brindamos una asesoría de que podría servirle para generar más ganancias.
- Venta anticipada de nuevos productos. Quizás haya nuevos productos que salen una determinada fecha y pueden llegar a ser un boom. Debemos darle la posibilidad a estos clientes de comprar estos productos de manera anticipada para que ellos también puedan venderlos el primer día y subirse a la moda que puede ser pasajera.

## Riesgos del análisis

El análisis de RFM realizado presenta varias limitaciones y riesgos que deben considerarse al interpretar los resultados. En primer lugar, si bien el enfoque RFM permite segmentar clientes en función de su comportamiento transaccional, se basa únicamente en tres dimensiones (Recency, Frequency, Monetary) y no captura otros aspectos relevantes como el tipo de productos adquiridos, la estacionalidad o la fidelidad a largo plazo. Esto se demuestra en los clusters desarrollados arriba, muchas veces había tipos de clientes mezclados que no tenían buena segmentación. 

Además, cabe recalcar que existe una gran correlación entre las variables del RFM (especialmente frequency y monetary), lo cual puede reducir la diversidad real de los clusters generados.

Además, al tratarse de una base de datos de un negocio desconocido, carecemos de contexto sobre la estrategia comercial o eventos que podrían haber afectado las ventas (como promociones, cambios logísticos, etc), lo cual puede sesgar la interpretación de los segmentos. Todas las interpretaciones de los clústeres y las etiquetas asignadas son hipotéticas; aunque lógicamente derivadas de los datos, requieren validación por parte de expertos con conocimiento del dominio.

Esta misma limitación se extiende a las propuestas de acciones de marketing, que son genéricas y deben ser adaptadas y evaluadas cuidadosamente según la infraestructura y objetivos operativos reales de la empresa.

Finalmente, la metodología de winsorization para los outliers introduce el riesgo de "aplanar" la información más extrema de los clientes de muy alto valor. Esto significa que los "Clientes VIP" identificados podrían ser incluso más valiosos de lo que se percibe a través de los datos capados, limitando la granularidad en la segmentación de los clientes de alto valor, lo que podría conducir a estrategias subóptimas si no se considera la verdadera escala de esos valores atípicos.